# Multivariate Imputation

Other features/variables are used/referred to fill the missing values of a given col. Two techiniques : 
1. KNN Imputer
2. Iterative Imputer

## 1. KNN Imputer
Uses KNN algo to choose the closes, or the most similar observation to impute the missing value.

### Working
1. The euclidean distance of the observation with the missing value is calculated with the all the other observations
2. The closest k neighbors are chosen
3. Value to impute is calculated

#### 1. Euclidean distance
Ex - there are 4 features (f1, f2, f3, f4)

Point P(x, 5, 6 ,8) is the observation we want to impute. So we calculate the euclidean distance of this observation using the other three variables with all the other observations. 

Let k = 2, then the top two observations are chosen. 


#### 2. Finding the value
The value to impute with = mean of the f1 values of the k nearest neighbors
#x (10, 6, 8, 7) and (3, 4 7, 10) are the 2 nearest observations. Then val = (10 + 3)/2 = 6 .

Hence P becomes p(6,5,6,8)

### Note : But it is possble that the other features' value are also missing at some observations. 

Thus to get around this problem, we use the nan_euclidean distance which is calculated as 
$$
\sqrt{(weight[(x1-x2)^2 + (y1-y2)^2...])}
$$
where  
$$
weight = \frac{\text{total num of features}}{\text{available num of features}} 
$$




### Advantages 
- More accurate than univariate cols


### Disadvantages 
- Too much calculation => slow
- Heavy to deply on server as the training dataset also needs to be uploaded along with the model

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('../../3_eda/titanic.csv', usecols=['Age', "Pclass", 'Fare', 'Survived'])
df.sample(5)


,Survived,Pclass,Age,Fare
185,0,1,NaN,50.000
613,0,3,NaN,7.750
880,1,2,25.0,26.000
816,0,3,23.0,7.925
743,0,3,24.0,16.100


In [4]:
df.isnull().mean()*100

Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64

In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train.head()

,Pclass,Age,Fare
331,1,45.5,28.5000
733,2,23.0,13.0000
382,3,32.0,7.9250
704,3,26.0,7.8542
813,3,6.0,31.2750


In [ ]:
# KNN Imputer object
## Experiment with the parameters - no. of neighbors, weights etc to find optimum imputation results
knn = KNNImputer()
X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)

In [7]:
# COnvert these to dataframes, as they are np arrays right now
pd.DataFrame(X_train_trf, columns=X_train.columns)

,Pclass,Age,Fare
0,1.0,45.5,28.5000
1,2.0,23.0,13.0000
2,3.0,32.0,7.9250
3,3.0,26.0,7.8542
4,3.0,6.0,31.2750
...,...,...,...
707,3.0,21.0,7.6500
708,1.0,39.0,31.0000
709,3.0,41.0,14.1083
710,1.0,14.0,120.0000


In [8]:
# Model
lr = LogisticRegression()

# training
lr.fit(X_train_trf, y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test, y_pred)

0.7430167597765364

- Change different parameters of the imputer and see how accuracy changes and try to find the optimum values of the parameters. 
- Can also compare the results with simpleIputer and see that is usually results in higher accuracy than the simple imputer strategies

## 2. Iterative Imputer - MICE (Multivariate Imputation by Chained Equations)
- Applied when variable with missing values is MAR (Missing At Random)

- Missing data in a variable can be either : 
    - MCAR (Missing Completely At Random) - when data for few observations, due to some reason could not be recorded. 
    - MAR (Missing At Random) - when the data was not willingly filled by the user, and is assumed to be predicted using imputation
    - MNAR (Missing Not At Random) - when the data is intentionally removed and the variable's cov with the other features is low so its missing values cannot be imputed using the other cols

- This Imputation is applied on data that is MAR. (and on the input cols only, obviously)

### Advantages :
- Accurate

### Disadvantages : 
- Slow due to large no. of calculations
- Memory heavy to deploy on server as the training data needs to be uploaded along with it for predictions in case of missing input data

### The MICE Algorithm 
Steps :
1. All missing values, in all variables are replaced by the means of their respective variables. 
2. first missing value is again replaced with nan
3. The rest of the observations are then used as training data for any chosen model to predict that missing value (the observation with the missing value acst as the test datset)
4. All the missing values are then one by one predicted in the same manner.
5. Once all originally missing values predicted at least once, that is the end of stage 1 and the first iteration.
6. Now all the values at the end of this stage are subtracted from the values after the mean imputation.
7. Then the iteration again begins with the prediction and subtraction from the previous stage/iteration till the difference for all the missing values reaches or becomes 0.
